# Riesgo de crédito de emisores de energía en Colombia — 02: análisis

Toma `tabla_emisores.csv` (cuaderno 01) y recorre los tres caminos de la prueba:
1. **Histórico**: ratios y figuras 2015–2025.
2. **Prospectivo (PD)**: scorecard de S&P Global Ratings (etapa 1: apalancamiento y cobertura; etapa 2: liquidez) → calificación ancla → probabilidad de incumplimiento (Berk & DeMarzo).
3. **Cuantitativo**: descomposición de la cobertura, correlaciones y prueba de estrés.

Salidas en Drive: `ratios_emisores.csv`, `liquidez_2025.csv`, `estres_2025.csv`, `resumen_2025.csv` y las figuras `fig*.png`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUTA = "/content/drive/MyDrive/Prueba_Bancolombia"
tabla = pd.read_csv(f"{RUTA}/tabla_emisores.csv")

emisores = ["ISA", "ISAGEN", "EPM", "CELSIA", "ENEL"]
# Paleta de colores fáciles de distinguir (apta para daltonismo)
colores = {"ISA": "#0072B2", "ISAGEN": "#E69F00", "EPM": "#009E73", "CELSIA": "#D55E00", "ENEL": "#7B3294"}
ANIOS = list(range(2015, 2026))

def ejes_anios(ax, rotar=False):
    """Marca todos los años 2015-2025 en el eje x."""
    ax.set_xticks(ANIOS)
    ax.set_xticklabels(ANIOS, rotation=90 if rotar else 0)
    ax.set_xlim(2014.5, 2025.5)

print(tabla.shape)
tabla.head()

## 1. Ratios
EBITDA = utilidad operacional + depreciación y amortización (misma fórmula para los cinco; no se usa el EBITDA que publica cada empresa).

In [ ]:
r = tabla.copy()
r["ebitda"] = r.utilidad_operacional + r.dya
r["margen_ebitda"] = r.ebitda / r.ingresos
r["margen_neto"] = r.utilidad_neta / r.ingresos
r["deuda_ebitda"] = r.deuda / r.ebitda
r["deuda_neta_ebitda"] = (r.deuda - r.caja) / r.ebitda
r["cobertura_intereses"] = r.ebitda / r.intereses
r["razon_corriente"] = r.activo_corriente / r.pasivo_corriente
r["caja_activos"] = r.caja / r.activos
r["cash_ratio"] = r.caja / r.pasivo_corriente
r["deuda_patrimonio"] = r.deuda / r.patrimonio
ratios = r.round(2)

cols = ["emisor", "ebitda", "margen_ebitda", "deuda_ebitda", "deuda_neta_ebitda", "cobertura_intereses", "razon_corriente", "cash_ratio"]
ratios[ratios.anio == 2025][cols].sort_values("deuda_ebitda")

## 2. Ajuste ISA 2016: ingreso extraordinario RBSE
En 2016 CTEEP (filial de ISA en Brasil) reconoció la indemnización de la RBSE (Red Básica del Sistema Existente): COP 5,5 billones de
ingreso no recurrente. Fuente: ISA, Reporte Integrado de Gestión 2016, EEFF consolidados, nota de cuentas por cobrar (p. 211).
Se resta de ingresos y EBITDA en las columnas `_aj`; la cifra reportada se conserva.

In [ ]:
RBSE_2016 = 5.5
ratios["ingresos_aj"] = ratios.ingresos
ratios["ebitda_aj"] = ratios.ebitda
m = (ratios.emisor == "ISA") & (ratios.anio == 2016)
ratios.loc[m, "ingresos_aj"] = ratios.loc[m, "ingresos"] - RBSE_2016
ratios.loc[m, "ebitda_aj"] = ratios.loc[m, "ebitda"] - RBSE_2016
ratios["deuda_ebitda_aj"] = ratios.deuda / ratios.ebitda_aj
ratios["cobertura_aj"] = ratios.ebitda_aj / ratios.intereses
ratios["margen_ebitda_aj"] = ratios.ebitda_aj / ratios.ingresos_aj
ratios.to_csv(f"{RUTA}/ratios_emisores.csv", index=False)
ratios[m][["emisor", "anio", "ingresos", "ingresos_aj", "ebitda", "ebitda_aj", "deuda_ebitda", "deuda_ebitda_aj", "cobertura_intereses", "cobertura_aj"]].round(2)

## 3. Camino histórico: evolución 2015–2025

In [ ]:
# Figura 1: EBITDA (barras) y deuda / EBITDA (línea) por emisor
fig, ejes = plt.subplots(1, 5, figsize=(22, 4.8))
tope = ratios.deuda_ebitda.max() * 1.15
orden = ratios[ratios.anio == 2025].sort_values("deuda_ebitda_aj").emisor.tolist()   # de menor a mayor apalancamiento
for ax, e in zip(ejes, orden):
    d = ratios[ratios.emisor == e].sort_values("anio")
    ax.bar(d.anio, d.ebitda, color="#C9D3E0", label="EBITDA (billones COP)")
    ax.set_title(e, fontweight="bold", color=colores[e])
    ax.set_ylabel("EBITDA (billones COP)")
    ax.set_ylim(0, d.ebitda.max() * 1.15)
    ejes_anios(ax, rotar=True)
    ax2 = ax.twinx()
    ax2.plot(d.anio, d.deuda_ebitda, color=colores[e], marker="o", linewidth=2.2, label="Deuda / EBITDA (x)")
    ax2.axhline(3, color="grey", linestyle="--", linewidth=1)
    ax2.set_ylim(0, tope)
    ax2.set_ylabel("Deuda / EBITDA (x)")
fig.suptitle("EBITDA (barras) y Deuda / EBITDA (línea), 2015–2025, de menor a mayor deuda/EBITDA en 2025. Línea gris: 3x. ISA 2016 sin ajustar",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{RUTA}/fig1_ebitda_deuda.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figura 2: cobertura de intereses por emisor
fig, ejes = plt.subplots(1, 5, figsize=(22, 4.2))
orden = ratios[ratios.anio == 2025].sort_values("cobertura_aj", ascending=False).emisor.tolist()   # de mayor a menor cobertura
for ax, e in zip(ejes, orden):
    d = ratios[ratios.emisor == e].sort_values("anio")
    ax.plot(d.anio, d.cobertura_aj, marker="o", linewidth=2.2, color=colores[e])
    ax.axhline(6, color="grey", linestyle=":", linewidth=1)      # frontera nivel 3
    ax.axhline(2, color="grey", linestyle="--", linewidth=1)     # frontera nivel 6
    ax.set_title(e, fontweight="bold", color=colores[e])
    ax.set_ylim(0, d.cobertura_aj.max() * 1.18)
    ax.set_ylabel("EBITDA / intereses (x)")
    ejes_anios(ax, rotar=True)
fig.suptitle("Cobertura de intereses por emisor, 2015\u20132025, de mayor a menor cobertura en 2025. L\u00edneas grises: 6x y 2x (umbrales S&P tabla 17)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{RUTA}/fig2_cobertura.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figura 3: razón corriente
fig, ax = plt.subplots(figsize=(11, 5))
for e in emisores:
    d = ratios[ratios.emisor == e].sort_values("anio")
    ax.plot(d.anio, d.razon_corriente, marker="o", linewidth=2.2, color=colores[e], label=e)
ax.axhline(1.0, color="grey", linestyle="--", linewidth=1)
ax.set_ylim(0, ratios.razon_corriente.max() * 1.12)
ejes_anios(ax)
ax.set_ylabel("Activo corriente / pasivo corriente (x)")
ax.set_title("Razón corriente, 2015–2025. Línea gris: 1,0x (activo corriente = pasivo corriente)", fontweight="bold")
ax.legend(loc="upper left", ncol=5, frameon=False)
plt.tight_layout()
plt.savefig(f"{RUTA}/fig3_razon_corriente.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Camino PD, etapa 1: scorecard de S&P → perfil financiero → calificación ancla → PD
Fuente: S&P Global Ratings, *Corporate Methodology*, 7 de enero de 2024, y *Sector-Specific Corporate Methodology*, 7 de julio de 2025.
Tabla 17 (volatilidad estándar) y tabla 18 (volatilidad media): límites de deuda/EBITDA por categoría; EBITDA/intereses como ratio
suplementario; tabla 3: matriz perfil de negocio × perfil financiero. PD por calificación: Berk & DeMarzo, *Corporate Finance*, cap. 12,
tabla 12.2 (fuente original Moody's).

El scorecard tiene dos entradas. El **perfil financiero** sale de los ratios (sección 4.2). El **perfil de negocio** se evalúa emisor por
emisor en la sección 4.1, siguiendo el paso 1 de la metodología, en vez de suponerlo igual para los cinco.

Decisión del analista declarada: tabla media para ISA, EPM y Enel (negocios regulados); estándar para ISAGEN y Celsia (generación
expuesta al clima y al precio de bolsa).

### 4.1 Perfil de negocio emisor por emisor
Se replica el paso 1 de la metodología de S&P: riesgo país × riesgo de industria → CICRA, y CICRA × posición competitiva → perfil de
negocio. La posición competitiva combina tres subfactores (ventaja regulatoria o competitiva, escala y diversificación, eficiencia
operativa) ponderados según el perfil de grupo del sector, ajustados por la rentabilidad (nivel × volatilidad medida con el error
estándar de la regresión del margen EBITDA contra el tiempo).

Los puntajes de los subfactores son juicio del analista; cada uno está sustentado con su fuente en `docs/perfil_negocio_fuentes.md`.
El perfil de negocio se evalúa una sola vez con los once años de historia y se aplica a toda la serie: S&P lo mira a través del ciclo,
no año por año.

Resultado: ISA y EPM satisfactorio, Enel fuerte, ISAGEN razonable, Celsia débil. Frente al supuesto de "satisfactorio para los cinco",
cambian dos: Enel sube y Celsia baja.

In [ ]:
# =====================================================================
# 4.bis  PERFIL DE NEGOCIO EMISOR POR EMISOR (S&P, paso 1 de la metodologia)
# Reemplaza el supuesto PERFIL_NEGOCIO = "satisfactory" para los cinco.
# Fuentes de cada puntaje: ver docs/perfil_negocio_fuentes.md
# =====================================================================

# ---- 1. Riesgo pais: ponderado por ingresos 2025 por area geografica -------
# S&P Country Risk Assessments Update, oct-2024: Colombia 4, Brasil 4, Peru 4,
# Chile 3, Panama 4, Guatemala 5, Costa Rica 4, El Salvador 6.
# Regla (Corporate Methodology parr. 22-26): paises con >5% de ventas, pesos
# redondeados al 5%, promedio redondeado al entero; si un pais pesa >=75%,
# el puntaje es el de ese pais.
RIESGO_PAIS = {"ISA": 4, "EPM": 4, "ENEL": 4, "CELSIA": 4, "ISAGEN": 4}

# ---- 2. Riesgo de industria: ponderado por EBITDA 2025 por segmento --------
# S&P Methodology: Industry Risk, Apendice IV tabla 7 (jul-2025):
# servicios publicos regulados 1, generacion no regulada 4, infraestructura
# de transporte 2. Se promedian las lineas con >20% del EBITDA (parr. 27).
RIESGO_INDUSTRIA = {"ISA": 1, "EPM": 2, "ENEL": 3, "CELSIA": 2, "ISAGEN": 4}

# ---- 3. CICRA (tabla 1 de la Corporate Methodology) ------------------------
TABLA_CICRA = {  # [riesgo_industria][riesgo_pais-1]
    1: [1, 1, 1, 2, 4, 5], 2: [2, 2, 2, 3, 4, 5], 3: [3, 3, 3, 3, 4, 6],
    4: [4, 4, 4, 4, 5, 6], 5: [5, 5, 5, 5, 5, 6], 6: [6, 6, 6, 6, 6, 6]}
CICRA = {e: TABLA_CICRA[RIESGO_INDUSTRIA[e]][RIESGO_PAIS[e] - 1] for e in emisores}

# ---- 4. Posicion competitiva ----------------------------------------------
# Subfactores en escala 1 (fuerte) a 5 (debil). Justificacion documentada.
VENTAJA    = {"ISA": 2, "EPM": 3, "CELSIA": 3, "ENEL": 2, "ISAGEN": 3}
ESCALA     = {"ISA": 1, "EPM": 2, "CELSIA": 4, "ENEL": 2, "ISAGEN": 4}
EFICIENCIA = {"ISA": 1, "EPM": 4, "CELSIA": 4, "ENEL": 2, "ISAGEN": 3}

# Perfil de grupo (CPGP) y pesos, Sector-Specific Corporate Methodology 2025:
# "National industry and utilities" 60/20/20 para los de mayoria regulada;
# "Capital or asset focus" 30/30/40 para generacion y verticalmente integrados.
PESOS = {"nacional_utilities": (0.60, 0.20, 0.20), "capital_activos": (0.30, 0.30, 0.40)}
CPGP  = {"ISA": "nacional_utilities", "EPM": "nacional_utilities",
         "CELSIA": "nacional_utilities", "ENEL": "capital_activos",
         "ISAGEN": "capital_activos"}

def cp_preliminar(x):                       # tabla 14
    for lim, v in [(1.50, 1), (2.25, 2), (3.00, 3), (3.75, 4), (4.50, 5), (5.01, 6)]:
        if x <= lim:
            return v

# ---- 5. Rentabilidad: nivel x volatilidad (SER) ---------------------------
# Volatilidad = error estandar de la regresion del margen EBITDA contra el
# tiempo, dividido por su promedio (Corporate Methodology parr. 83-84).
SER_CORTES = {"regulada": [3, 5, 7, 11, 21], "no_regulada": [5, 10, 14, 21, 39]}
SECTOR_SER = {"ISA": "regulada", "EPM": "regulada", "CELSIA": "regulada",
              "ENEL": "no_regulada", "ISAGEN": "no_regulada"}
NIVEL_RENT = {"ISA": "arriba", "EPM": "promedio", "CELSIA": "abajo",
              "ENEL": "arriba", "ISAGEN": "arriba"}
TABLA15 = {"arriba": [1, 1, 2, 3, 4, 5], "promedio": [1, 2, 3, 4, 5, 6],
           "abajo":  [2, 3, 4, 5, 6, 6]}
TABLA16 = {1: [1, 2, 2, 3, 4, 5], 2: [1, 2, 3, 3, 4, 5], 3: [2, 2, 3, 4, 4, 5],
           4: [2, 3, 3, 4, 5, 5], 5: [2, 3, 4, 4, 5, 6], 6: [2, 3, 4, 5, 5, 6]}

def ser_margen(e):
    d = ratios[ratios.emisor == e].sort_values("anio")
    y = (d.margen_ebitda_aj * 100).values
    x = d.anio.values.astype(float)
    b, a = np.polyfit(x, y, 1)
    se = np.sqrt(((y - (a + b * x)) ** 2).sum() / (len(y) - 2))
    return se / y.mean() * 100

def nivel_vol(ser, cortes):
    for i, c in enumerate(cortes):
        if ser <= c:
            return i + 1
    return 6

# ---- 6. Perfil de negocio (tabla 2) ---------------------------------------
TABLA2 = {1: [1, 1, 1, 2, 3, 5], 2: [1, 2, 2, 3, 4, 5], 3: [2, 3, 3, 3, 4, 6],
          4: [3, 4, 4, 4, 5, 6], 5: [4, 5, 5, 5, 5, 6], 6: [5, 6, 6, 6, 6, 6]}
NOMBRE_NEG = {1: "excellent", 2: "strong", 3: "satisfactory",
              4: "fair", 5: "weak", 6: "vulnerable"}

detalle = []
for e in emisores:
    ser = ser_margen(e)
    vol = nivel_vol(ser, SER_CORTES[SECTOR_SER[e]])
    rent = TABLA15[NIVEL_RENT[e]][vol - 1]
    w = PESOS[CPGP[e]]
    prom = w[0] * VENTAJA[e] + w[1] * ESCALA[e] + w[2] * EFICIENCIA[e]
    prel = cp_preliminar(prom)
    cp = TABLA16[rent][prel - 1]
    neg = TABLA2[cp][CICRA[e] - 1]
    detalle.append({"emisor": e, "SER_%": round(ser, 1), "volatilidad": vol,
                    "rentabilidad": rent, "prom_subfactores": round(prom, 2),
                    "pos_comp_prelim": prel, "pos_comp": cp, "CICRA": CICRA[e],
                    "perfil_negocio": NOMBRE_NEG[neg]})

perfil_negocio_tabla = pd.DataFrame(detalle).set_index("emisor")
print(perfil_negocio_tabla.to_string())

# Esto es lo que consume el scorecard: reemplaza el supuesto "satisfactory".
PERFIL_NEGOCIO = perfil_negocio_tabla.perfil_negocio.to_dict()

# La matriz de anclas necesita las filas nuevas (fuerte, razonable, debil)
MATRIZ = {"excellent":    ["aaa/aa+", "aa", "a+/a", "a-", "bbb", "bbb-/bb+"],
          "strong":       ["aa/aa-", "a+/a", "a-/bbb+", "bbb", "bb+", "bb"],
          "satisfactory": ["a/a-", "bbb+", "bbb/bbb-", "bbb-/bb+", "bb", "b+"],
          "fair":         ["bbb/bbb-", "bbb-", "bb+", "bb", "bb-", "b"],
          "weak":         ["bb+", "bb+", "bb", "bb-", "b+", "b/b-"],
          "vulnerable":   ["bb-", "bb-", "bb-/b+", "b+", "b", "b-"]}

### 4.2 Perfil financiero, calificación ancla y PD

In [ ]:
DESCRIPTORES = ["minimal", "modest", "intermediate", "significant", "aggressive", "highly leveraged"]
DE_ESTANDAR = [1.5, 2.0, 3.0, 4.0, 5.0]        # deuda/EBITDA: <1.5, 1.5-2, 2-3, 3-4, 4-5, >5
DE_MEDIA    = [1.75, 2.5, 3.5, 4.5, 5.5]       # deuda/EBITDA: <1.75, 1.75-2.5, 2.5-3.5, 3.5-4.5, 4.5-5.5, >5.5
COBERTURA   = [15, 10, 6, 3, 2]                # EBITDA/intereses: >=15, 10-15, 6-10, 3-6, 2-3, <2
# MATRIZ (tabla 3 completa, seis filas) y PERFIL_NEGOCIO vienen de la sección 4.1.
TABLA_DE = {"ISA": DE_MEDIA, "EPM": DE_MEDIA, "ENEL": DE_MEDIA, "ISAGEN": DE_ESTANDAR, "CELSIA": DE_ESTANDAR}

def desc_deuda(v, limites):
    for i, l in enumerate(limites):
        if v < l: return i + 1
    return 6

def desc_cobertura(v):
    for i, l in enumerate(COBERTURA):
        if v >= l: return i + 1
    return 6

def perfil_financiero(de, cob, limites):
    """El ratio núcleo (deuda/EBITDA) manda; el suplementario (cobertura) mueve una categoría si difiere en 2 o más."""
    d, c = desc_deuda(de, limites), desc_cobertura(cob)
    ajuste = 1 if c - d >= 2 else (-1 if d - c >= 2 else 0)
    return max(1, min(6, d + ajuste))

# PD anual promedio por calificación (Berk & DeMarzo, tabla 12.2; fuente Moody's)
PD_RATING = {"aa": 0.001, "a": 0.002, "bbb": 0.005, "bb": 0.022, "b": 0.055}
def pd_de_ancla(ancla):
    letra = ancla.split("/")[0].rstrip("+-")          # "bbb/bbb-" -> "bbb"
    return PD_RATING[letra] * 100

ratios["nivel_deuda"] = [desc_deuda(de, TABLA_DE[e]) for e, de in zip(ratios.emisor, ratios.deuda_ebitda_aj)]
ratios["nivel_cobertura"] = ratios.cobertura_aj.apply(desc_cobertura)
ratios["perfil_fin"] = [perfil_financiero(de, cob, TABLA_DE[e]) for e, de, cob in zip(ratios.emisor, ratios.deuda_ebitda_aj, ratios.cobertura_aj)]
ratios["perfil_fin_nombre"] = ratios.perfil_fin.apply(lambda i: DESCRIPTORES[i - 1])
ratios["ancla_sp"] = [MATRIZ[PERFIL_NEGOCIO[e]][i - 1] for e, i in zip(ratios.emisor, ratios.perfil_fin)]
ratios["perfil_negocio"] = ratios.emisor.map(PERFIL_NEGOCIO)
# Comparación: qué habría dado el supuesto de "perfil de negocio satisfactorio para los cinco"
ratios["ancla_supuesto"] = [MATRIZ["satisfactory"][i - 1] for i in ratios.perfil_fin]
ratios["pd_sp_pct"] = ratios.ancla_sp.apply(pd_de_ancla)
ratios["pd_supuesto_pct"] = ratios.ancla_supuesto.apply(pd_de_ancla)
ratios.to_csv(f"{RUTA}/ratios_emisores.csv", index=False)

cols = ["emisor", "deuda_ebitda_aj", "nivel_deuda", "cobertura_aj", "nivel_cobertura", "perfil_fin", "perfil_fin_nombre",
        "perfil_negocio", "ancla_supuesto", "pd_supuesto_pct", "ancla_sp", "pd_sp_pct"]
ratios[ratios.anio == 2025].sort_values("perfil_fin")[cols].round(2)

In [ ]:
# Figura 4: perfil de riesgo financiero por año (1 = mejor, 6 = peor)
fig, ax = plt.subplots(figsize=(11, 5))
for e in emisores:
    d = ratios[ratios.emisor == e].sort_values("anio")
    ax.plot(d.anio, d.perfil_fin, marker="o", linewidth=2.2, color=colores[e], label=e)
ax.set_yticks(range(1, 7)); ax.set_yticklabels([f"{i} {n}" for i, n in enumerate(DESCRIPTORES, 1)])
ax.set_ylim(6.6, 0.4)
ejes_anios(ax)
ax.set_title("Perfil de riesgo financiero S&P por año (1 = mejor, 6 = peor)", fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=5, frameon=False)
plt.tight_layout()
plt.savefig(f"{RUTA}/fig4_perfil_sp.png", dpi=150, bbox_inches="tight")
plt.show()

print("Calificación ancla por año:")
ratios.pivot(index="anio", columns="emisor", values="ancla_sp")[emisores]

In [ ]:
# Figura 5: mapa 2025, apalancamiento vs. cobertura sobre las zonas de S&P (tabla 17)
d25 = ratios[ratios.anio == 2025].set_index("emisor").loc[emisores]
fig, ax = plt.subplots(figsize=(9, 6))
for x in DE_ESTANDAR: ax.axvline(x, color="lightgrey", linestyle="--", linewidth=1)
for y in [6, 3, 2]: ax.axhline(y, color="lightgrey", linestyle=":", linewidth=1)
for e in emisores:
    ax.scatter(d25.loc[e, "deuda_ebitda_aj"], d25.loc[e, "cobertura_aj"], s=180, color=colores[e], zorder=3, label=e)
    ax.annotate(e, (d25.loc[e, "deuda_ebitda_aj"], d25.loc[e, "cobertura_aj"]), xytext=(8, 6), textcoords="offset points", fontweight="bold", color=colores[e])
ax.set_xlim(0, d25.deuda_ebitda_aj.max() * 1.25); ax.set_ylim(0, d25.cobertura_aj.max() * 1.2)
ax.set_xlabel("Deuda / EBITDA (x)  —  líneas verticales: límites S&P tabla 17 (1,5 · 2 · 3 · 4 · 5)")
ax.set_ylabel("EBITDA / intereses (x)  —  líneas horizontales: 6 · 3 · 2")
ax.set_title("Mapa 2025: apalancamiento vs. cobertura sobre las zonas de S&P", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{RUTA}/fig5_mapa_sp.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Camino PD, etapa 2: prueba de liquidez de S&P (fuentes y usos de caja)
Fuente: S&P Global Ratings, *Methodology And Assumptions: Liquidity Descriptors For Global Corporate Issuers*, 16 de diciembre de 2014.
- Fuentes (A): caja + FFO (EBITDA − intereses pagados − impuestos pagados) + líneas de crédito **comprometidas** no usadas.
- Usos (B): deuda que vence en 12 meses + capex + dividendos.
- Descriptores: A/B ≥ 2,0 excepcional | ≥ 1,5 fuerte | ≥ 1,2 adecuada | < 1,2 menos que adecuada | déficit (< 1,0) débil, con prueba de
  resistencia (A − B positivo si el EBITDA cae 50% / 30% / 15%).
- Efecto sobre la ancla (Corporate Methodology, párr. 36): adecuada o mejor no cambia; menos que adecuada, tope bb+; débil, tope b-.

Ningún emisor revela líneas comprometidas en sus EEFF 2025 → la prueba se corre sin líneas (conservadora) y el tope se reporta como sensibilidad.
Los vencimientos del año 2 vienen de las notas de los PDF de cierre 2025 (ISA nota 17.4; ISAGEN nota 19; Celsia nota de obligaciones;
EPM nota 44.5, capital + intereses; Enel riesgo de liquidez, capital + intereses).

In [ ]:
ratios["ffo"] = ratios.ebitda_aj - ratios.intereses_pagados - ratios.impuestos_pagados
ratios["fuentes_12m"] = ratios.caja + ratios.ffo
ratios["usos_12m"] = ratios.deuda_corto_plazo + ratios.capex + ratios.dividendos_pagados
ratios["liq_ratio"] = ratios.fuentes_12m / ratios.usos_12m

def descriptor(fuentes, usos, ebitda):
    r = fuentes / usos
    if r >= 2.0 and fuentes - 0.50 * ebitda - usos > 0: return "excepcional"
    if r >= 1.5 and fuentes - 0.30 * ebitda - usos > 0: return "fuerte"
    if r >= 1.2 and fuentes - 0.15 * ebitda - usos > 0: return "adecuada"
    if r >= 1.0: return "menos que adecuada"
    return "débil"

ratios["liquidez"] = [descriptor(f, u, e) for f, u, e in zip(ratios.fuentes_12m, ratios.usos_12m, ratios.ebitda_aj)]

ESCALA = ["aa", "aa-", "a+", "a", "a-", "bbb+", "bbb", "bbb-", "bb+", "bb", "bb-", "b+", "b", "b-"]
def con_tope(ancla, liquidez):
    nota = ancla.split("/")[0]
    tope = {"menos que adecuada": "bb+", "débil": "b-"}.get(liquidez)
    if tope is None or ESCALA.index(nota) >= ESCALA.index(tope): return ancla
    return tope + " (tope liquidez)"
ratios["ancla_con_liquidez"] = [con_tope(a, l) for a, l in zip(ratios.ancla_sp, ratios.liquidez)]
ratios.to_csv(f"{RUTA}/ratios_emisores.csv", index=False)

# 2025: prueba a 24 meses y sensibilidades
DEUDA_ANIO2 = {"ISA": 1.78, "ISAGEN": 1.26, "CELSIA": 3.35 / 4, "EPM": 7.62, "ENEL": 3.92 / 2}   # billones, de los PDF 2025
LINEAS_NO_COMPROMETIDAS = {"ENEL": 4.52}   # Enel revela $4,52 bn "autorizadas no utilizadas", sujetas a nueva aprobación

l25 = ratios[ratios.anio == 2025].set_index("emisor").loc[emisores].copy()
l25["deuda_anio2"] = pd.Series(DEUDA_ANIO2)
l25["fuentes_24m"] = l25.caja + 2 * l25.ffo
l25["usos_24m"] = l25.deuda_corto_plazo + l25.deuda_anio2 + 2 * (l25.capex + l25.dividendos_pagados)
l25["liq_ratio_24m"] = l25.fuentes_24m / l25.usos_24m
l25["resiste_ebitda_-15%"] = (l25.fuentes_12m - 0.15 * l25.ebitda_aj - l25.usos_12m) > 0
l25["liq_ratio_con_lineas"] = (l25.fuentes_12m + l25.index.map(LINEAS_NO_COMPROMETIDAS).fillna(0)) / l25.usos_12m
l25["liq_ratio_sin_dividendos"] = l25.fuentes_12m / (l25.usos_12m - l25.dividendos_pagados)

cols = ["caja", "ffo", "fuentes_12m", "deuda_corto_plazo", "capex", "dividendos_pagados", "usos_12m", "liq_ratio",
        "liq_ratio_24m", "resiste_ebitda_-15%", "liquidez", "razon_corriente", "cash_ratio", "ancla_sp", "ancla_con_liquidez",
        "liq_ratio_con_lineas", "liq_ratio_sin_dividendos"]
liquidez_2025 = l25[cols].round(2)
liquidez_2025.to_csv(f"{RUTA}/liquidez_2025.csv")
liquidez_2025.T

In [ ]:
# Figura 6: fuentes / usos de caja a 12 meses por año
fig, ax = plt.subplots(figsize=(11, 5))
for e in emisores:
    d = ratios[ratios.emisor == e].sort_values("anio")
    ax.plot(d.anio, d.liq_ratio, marker="o", linewidth=2.2, color=colores[e], label=e)
ax.axhline(1.2, color="grey", linestyle="--", linewidth=1)
ax.axhline(1.0, color="grey", linestyle=":", linewidth=1)
ax.set_ylim(0, ratios.liq_ratio.max() * 1.12)
ejes_anios(ax)
ax.set_ylabel("Fuentes / usos de caja a 12 meses (x)")
ax.set_title("Prueba de liquidez S&P por año, 2015–2025 (sin líneas comprometidas). Líneas grises: 1,2x adecuada y 1,0x fuentes = usos", fontweight="bold")
ax.legend(loc="upper left", ncol=5, frameon=False)
plt.tight_layout()
plt.savefig(f"{RUTA}/fig6_liquidez.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Camino cuantitativo: relación entre variables
1. **Descomposición exacta** de la cobertura: EBITDA/intereses = 1 / (deuda/EBITDA × tasa implícita), con tasa implícita = intereses / deuda.
2. **Correlaciones de Spearman** entre los indicadores sobre las 55 observaciones.
3. La **prueba de estrés** (sección 7) es el análisis de sensibilidad.

In [ ]:
ratios["tasa_implicita"] = ratios.intereses / ratios.deuda
ratios["cobertura_descompuesta"] = 1 / (ratios.deuda_ebitda_aj * ratios.tasa_implicita)   # debe coincidir con cobertura_aj
ratios.to_csv(f"{RUTA}/ratios_emisores.csv", index=False)

d25 = ratios[ratios.anio == 2025].set_index("emisor").loc[emisores]
print("Descomposición 2025: cobertura = 1 / (deuda/EBITDA x tasa implícita)")
print(pd.DataFrame({"deuda_ebitda": d25.deuda_ebitda_aj, "tasa_implicita_%": d25.tasa_implicita * 100,
                    "cobertura": d25.cobertura_aj, "verificacion": d25.cobertura_descompuesta}).round(2).to_string())

cols = ["deuda_ebitda_aj", "tasa_implicita", "cobertura_aj", "cash_ratio", "razon_corriente", "liq_ratio", "perfil_fin"]
print("\nCorrelación de Spearman entre indicadores, 55 observaciones:")
ratios[cols].corr(method="spearman").round(2)

## 7. Prueba de estrés 2025 (análisis de sensibilidad)
Golpes calibrados con la propia serie: EBITDA −20% (El Niño: en 2015-2016 el EBITDA de ISAGEN cayó 30%), intereses +30% (a EPM le
subieron 43% entre 2022 y 2023), y ambos. Alertas: deuda/EBITDA > 4,5x o cobertura < 2x. El golpe de El Niño no aplica a ISA (transmisión regulada).

In [ ]:
base = ratios[ratios.anio == 2025].set_index("emisor").loc[emisores]
escenarios = {
    "Base 2025":               dict(ebitda=1.00, intereses=1.00),
    "EBITDA -20% (El Niño)":   dict(ebitda=0.80, intereses=1.00),
    "Intereses +30% (tasas)":  dict(ebitda=1.00, intereses=1.30),
    "Ambos golpes":            dict(ebitda=0.80, intereses=1.30),
}
filas = []
for nombre, s in escenarios.items():
    for e in emisores:
        ebitda = base.loc[e, "ebitda_aj"] * s["ebitda"]
        intereses = base.loc[e, "intereses"] * s["intereses"]
        filas.append({"escenario": nombre, "emisor": e, "deuda_ebitda": base.loc[e, "deuda"] / ebitda, "cobertura": ebitda / intereses})
estres = pd.DataFrame(filas).round(2)
estres["alerta"] = (estres.deuda_ebitda > 4.5) | (estres.cobertura < 2)
estres.to_csv(f"{RUTA}/estres_2025.csv", index=False)

orden = list(escenarios)
pd.concat([estres.pivot(index="emisor", columns="escenario", values="cobertura").loc[emisores, orden].add_prefix("cobertura | "),
           estres.pivot(index="emisor", columns="escenario", values="deuda_ebitda").loc[emisores, orden].add_prefix("deuda/EBITDA | ")], axis=1)

In [ ]:
# Figura 7: cobertura 2025, base vs. ambos golpes
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(emisores)); w = 0.38
b = estres[estres.escenario == "Base 2025"].set_index("emisor").loc[emisores, "cobertura"]
g = estres[estres.escenario == "Ambos golpes"].set_index("emisor").loc[emisores, "cobertura"]
ax.bar(x - w/2, b, w, color="#C9D3E0", label="Base 2025")
ax.bar(x + w/2, g, w, color="#1F3B73", label="EBITDA -20% e intereses +30%")
ax.axhline(2, color="grey", linestyle="--", linewidth=1); ax.text(-0.45, 2.08, "umbral 2x", color="grey", fontsize=9, ha="left")
ax.set_xticks(x); ax.set_xticklabels(emisores)
ax.set_ylim(0, b.max() * 1.30)
ax.set_ylabel("EBITDA / intereses (x)")
ax.set_title("Prueba de estrés: cobertura de intereses 2025", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{RUTA}/fig7_estres.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Contraste externo: el modelo Z de Altman
El scorecard de S&P mide **flujo** (cuánto EBITDA genera la empresa contra cuánta deuda tiene). El modelo de Altman mide
**estructura de balance**. Son dos preguntas distintas, así que no tienen por qué coincidir — y donde no coinciden está lo
interesante.

No se estima un modelo nuevo: los coeficientes de Altman están publicados y se aplican tal cual. Antes de aplicarlos se
reproduce la pantalla `AZS` de Bloomberg con sus propios insumos, para saber exactamente qué versión del modelo usa.

In [ ]:
# La pantalla AZS de Bloomberg (Celsia) reporta dos números y no dice cuál es cuál.
# Con los insumos de la propia pantalla se reproducen exactamente, así que sabemos
# qué modelo usa cada uno:
#   "Z-score de Altman"             0,85  -> modelo original de 1968, cinco variables
#   "Z-score de Altman doble Prime" 0,70  -> modelo Z'' de 1995, cuatro variables
# Dos detalles que no son evidentes:
#   1. el denominador es ACTIVOS TANGIBLES (activo total - plusvalía - intangibles)
#   2. el Z'' que muestra Bloomberg NO incluye la constante +3,25 de la versión
#      para mercados emergentes (Z''-EM), que es la que tiene zonas de corte.

# ---- Validación: reproducir la pantalla con los insumos de Bloomberg -------
# Celsia, período fiscal 2026 Q2, cifras en millones de COP tal como aparecen.
bb = dict(at=14_446_227, fm=-569_941.94, br=-57_396.84, ebit=1_012_393.75,
          vm=5_247_211.5, pas=10_095_716, vta=0.36, cap=4_829_511.5)
z_1968 = (1.2 * bb["fm"] / bb["at"] + 1.4 * bb["br"] / bb["at"] + 3.3 * bb["ebit"] / bb["at"]
          + 0.6 * bb["vm"] / bb["pas"] + 1.0 * bb["vta"])
z_pp = (6.56 * bb["fm"] / bb["at"] + 3.26 * bb["br"] / bb["at"]
        + 6.72 * bb["ebit"] / bb["at"] + 1.05 * bb["cap"] / bb["pas"])
print(f"Validación con los insumos de Bloomberg: Z = {z_1968:.4f} (pantalla 0,85) | Z'' = {z_pp:.4f} (pantalla 0,70)")

# ---- El mismo cálculo, con nuestros XBRL, para los cinco y once años -------
ratios["at_tang"] = ratios.activos - ratios.goodwill - ratios.intangibles
ratios["X1"] = (ratios.activo_corriente - ratios.pasivo_corriente) / ratios.at_tang   # capital de trabajo
ratios["X2"] = ratios.utilidades_retenidas / ratios.at_tang                           # utilidades retenidas
ratios["X3"] = ratios.utilidad_operacional / ratios.at_tang                           # EBIT
ratios["X4"] = ratios.patrimonio / ratios.pasivo_total                                # patrimonio en libros
ratios["z_pp"] = 6.56 * ratios.X1 + 3.26 * ratios.X2 + 6.72 * ratios.X3 + 1.05 * ratios.X4
ratios["z_pp_em"] = ratios.z_pp + 3.25          # escala de mercados emergentes (Altman, Hartzell y Peck, 1995)

# El modelo de 1968 necesita el valor de mercado del patrimonio. Solo ISA y Celsia
# cotizan; EPM, ISAGEN y Enel Colombia no tienen acción, así que para ellos ese
# modelo no se puede calcular. Se deja vacío a propósito: no se estima.
CAP_MERCADO_2025 = {}     # {"ISA": x, "CELSIA": y} en billones de COP, al 31-dic-2025

# ---- Equivalencia con calificaciones y PD ---------------------------------
# Medianas de Z''-EM por calificación (Altman y Hotchkiss, tablas de bond rating
# equivalents; medianas de 2013). Se asigna la calificación más cercana.
EQUIV_Z = [("AAA/AA+", 8.80), ("AA/AA-", 8.40), ("A+", 8.22), ("A", 6.94), ("A-", 6.12),
           ("BBB+", 5.80), ("BBB", 5.75), ("BBB-", 5.70), ("BB+", 5.65), ("BB", 5.52),
           ("BB-", 5.07), ("B+", 4.81), ("B", 4.03), ("B-", 3.74), ("CCC+", 2.84),
           ("CCC", 2.57), ("CCC-", 1.72), ("CC/D", 0.05)]

def equiv_altman(z):
    return min(EQUIV_Z, key=lambda par: abs(par[1] - z))[0]

def zona_altman(z):                          # zonas de Z''-EM
    return "segura" if z > 2.6 else ("gris" if z >= 1.1 else "distress")

a25 = ratios[ratios.anio == 2025].set_index("emisor").loc[emisores].copy()
a25["equiv"] = a25.z_pp_em.apply(equiv_altman)
a25["pd_altman_%"] = a25.equiv.str.split("/").str[0].str.rstrip("+-").str.lower().map(PD_RATING) * 100
a25["zona"] = a25.z_pp_em.apply(zona_altman)

cols = ["X1", "X2", "X3", "X4", "z_pp", "z_pp_em", "equiv", "zona", "pd_altman_%"]
altman_2025 = a25[cols].round(3).sort_values("z_pp_em", ascending=False)
altman_2025.to_csv(f"{RUTA}/altman_2025.csv")
print("Las zonas (>2,6 segura) y la equivalencia con calificaciones son dos lecturas del mismo Z''-EM:")
print("la de zonas es permisiva, la de calificaciones es exigente. Por eso Celsia sale 'segura' y a la vez B.")
altman_2025


## 9. Resumen 2025

In [ ]:
resumen = ratios[ratios.anio == 2025].set_index("emisor").loc[emisores]
resumen = pd.DataFrame({
    "deuda_ebitda": resumen.deuda_ebitda_aj, "cobertura": resumen.cobertura_aj, "tasa_implicita_%": resumen.tasa_implicita * 100,
    "perfil_financiero": resumen.perfil_fin_nombre, "perfil_negocio": resumen.perfil_negocio,
    "ancla_supuesto": resumen.ancla_supuesto, "pd_supuesto_%": resumen.pd_supuesto_pct,
    "ancla_sp": resumen.ancla_sp, "pd_1_anio_%": resumen.pd_sp_pct,
    "fuentes_usos_12m": resumen.liq_ratio, "liquidez": resumen.liquidez, "cash_ratio": resumen.cash_ratio,
    "cobertura_ambos_golpes": estres[estres.escenario == "Ambos golpes"].set_index("emisor").loc[emisores, "cobertura"],
}).round(2).sort_values(["pd_1_anio_%", "perfil_financiero"])
resumen.to_csv(f"{RUTA}/resumen_2025.csv")
resumen